# 🚢 Exploratory Data Analysis & Feature Engineering
### Maritime Navigation Exercise Complexity Classifier

This notebook demonstrates:
1. **Class Distribution Analysis**: Distribution of target exercise difficulties (`LOW`, `MEDIUM`, `HIGH`).
2. **Ordinal Feature Engineering**: Mapping encounter situations to monotonic physical risk (`LOW=1.0 < MEDIUM=2.0 < HIGH=3.0`).
3. **Feature Correlation Matrix**: Aggregates (`mean`, `max`, `min`, `count`) vs. target complexity.
4. **Feature Importance in XGBoost**: Relative importance of aggregated risk features.

In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import json
from xgboost import XGBClassifier

# Paths
DATA_DIR = Path("../../data/labeled")
ASSETS_DIR = Path("../../model_assets")

# Load labeled CSV files
csv_files = list(DATA_DIR.glob("*.csv"))
print(f"Found {len(csv_files)} labeled exercise CSVs")
df_list = [pd.read_csv(f) for f in csv_files]
df = pd.concat(df_list, ignore_index=True)
df.head()

## 1. Distribution of Individual Situations vs Exercises

In [ ]:
# Normalize categories to canonical English terms
spanish_to_english = {"BAJA": "LOW", "MEDIA": "MEDIUM", "ALTA": "HIGH"}
df["SituationDifficulty"] = df["DificultadSituacion"].map(spanish_to_english)
df["ExerciseDifficulty"] = df["DificultadEjercicio"].map(spanish_to_english)

# Count situation distribution
situation_counts = df["SituationDifficulty"].value_counts()
print("Situation Risk Encounters:")
print(situation_counts)

# Aggregate per exercise
exercise_df = df[["Ejercicio", "ExerciseDifficulty"]].drop_duplicates()
exercise_counts = exercise_df["ExerciseDifficulty"].value_counts()
print("\nOverall Exercise Difficulties:")
print(exercise_counts)

## 2. Ordinal Feature Engineering
We assign physical monotonic risk values: `LOW=1.0, MEDIUM=2.0, HIGH=3.0`.

In [ ]:
ordinal_map = {"LOW": 1, "MEDIUM": 2, "HIGH": 3}
df["SituationOrdinal"] = df["SituationDifficulty"].map(ordinal_map)

# Compute 4 aggregated features per exercise
agg_df = df.groupby("Ejercicio").agg(
    mean_difficulty=("SituationOrdinal", "mean"),
    max_difficulty=("SituationOrdinal", "max"),
    min_difficulty=("SituationOrdinal", "min"),
    situation_count=("SituationOrdinal", "count")
).reset_index()

dataset = pd.merge(agg_df, exercise_df, on="Ejercicio")
dataset.describe()

## 3. Feature Importance from Native XGBoost Booster

In [ ]:
# Load trained native model
model_path = ASSETS_DIR / "model.json"
if model_path.exists():
    import xgboost as xgb
    booster = xgb.Booster()
    booster.load_model(str(model_path))
    
    # Feature importance based on weight / gain
    score_weight = booster.get_score(importance_type="weight")
    score_gain = booster.get_score(importance_type="gain")
    print("Feature Importance (Weight / Frequency):")
    print(score_weight)
    print("\nFeature Importance (Gain / Information Value):")
    print(score_gain)
else:
    print("Trained model.json not found in model_assets/. Run python ml_pipeline/train_model.py first.")